[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlatcl/multilogue/blob/main/notebooks/01_assess.ipynb)

# 01 · Assess — work you can do *without* the question

Assess is deliberately narrow. In the Fynesse framing, it covers only tasks that remain valid if the downstream research question changes: sizes, emptiness, crude vocabulary, coverage across documents. Anything that needs a concern/benefit taxonomy, a framing lens, or a headline metric belongs in **Address**.

**Why this split?** Reusable quality work should not be locked inside a question-specific notebook. Assess helpers return tables; plots here are diagnostics, not findings.

In [ ]:
#@title Setup (run first)
# Free path: synthetic units + FakeClient (no API key).
# Optional: load("polis", ...) downloads public open data into a local cache.

import sys, subprocess, pathlib

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "matplotlib", "pandas", "numpy", "scikit-learn", "pyyaml",
        "git+https://github.com/mlatcl/multilogue.git",
    ])
else:
    try:
        import multilogue  # noqa: F401
    except ImportError:
        _here = pathlib.Path.cwd().resolve()
        for _cand in (_here, _here.parent, *_here.parents):
            if (_cand / "pyproject.toml").is_file() and (_cand / "multilogue").is_dir():
                if str(_cand) not in sys.path:
                    sys.path.insert(0, str(_cand))
                break

from IPython.display import display, HTML, Markdown
from multilogue.notebook import setup

FIG = setup(figures_dir="outputs/notebook_figures")
print("Figures →", FIG)


In [ ]:
from multilogue.access import load
from multilogue.assess_base import (
    unit_size_table, docs_per_group, empty_unit_flags, token_frequency,
)
from multilogue import viz

units = load("synthetic", n_docs_per_group=4, units_per_doc=3)["units"]
sizes = unit_size_table(units)
empty = empty_unit_flags(units, min_words=4)
print(f"units={len(units)}; below min_words={len(empty)}")
display(docs_per_group(units, "group"))
display(token_frequency([u["text"] for u in units], top_n=15))

## Diagnostics

Unit-size histograms catch OCR debris and truncated chunks before any model call. Document counts by group show whether later *document-weighted* comparisons will rest on a thin sample.

In [ ]:
viz.unit_size_hist(sizes, group_col="group", units=units, path=FIG / "01_unit_sizes.png")
viz.docs_per_group_bars(docs_per_group(units, "group"), path=FIG / "01_docs.png")